# 1. Setup & Load YOLOv11 Model

In [1]:

import os
import sys
import cv2
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Suppress YOLO verbose output
import logging
logging.getLogger('ultralytics').setLevel(logging.ERROR)

# Import local object risk score modules
import risk_score_calculator
import trajectory_planner

model_path = '/Users/pavanreddy/Epilogue/vehicle_behaviour_pipeline/best_parinita.pt'
print(f'Loading YOLO Model from {model_path}...')
model = YOLO(model_path)
print('✅ Model Loaded Successfully')


Loading YOLO Model from /Users/pavanreddy/Epilogue/YOLO_Inference/best_parinita.pt...
✅ Model Loaded Successfully


# 2. End-to-End Sensor Fusion & Control Pipeline

In [2]:
input_video = '/Users/pavanreddy/Epilogue/YOLO_Inference/matlab_data_4/cam_video.mp4'
cap = cv2.VideoCapture(input_video)

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0: fps = 30.0
dt = 1.0 / fps
ego_speed = 10.0 # m/s

timetable_records = []
fusion_logs = []

frame_idx = 0
print('Starting pipeline...')

# Create a figure for plotting inline frames
from IPython.display import display, clear_output
fig, ax = plt.subplots(figsize=(10, 6))

while True:
    ret, frame = cap.read()
    if not ret: break
    
    # 1. YOLO Inference
    results = model.predict(frame, conf=0.25, verbose=False)
    boxes = results[0].boxes
    
    world_objects = []
    
    # 2. Extract Objects & Create Dynamic Radar Mock
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls_id = int(box.cls[0].cpu().numpy())
        
        w = x2 - x1
        mock_dist = max(3.0, 1500.0 / (w + 1.0)) 
        img_center_x = frame.shape[1] / 2
        obj_center_x = (x1 + x2) / 2
        mock_azimuth = (obj_center_x - img_center_x) / img_center_x * 30.0 
        mock_rate = -5.0 
        
        azimuth_rad = math.radians(mock_azimuth)
        x_current = mock_dist * math.cos(azimuth_rad)
        y_current = mock_dist * math.sin(azimuth_rad)
        vx = mock_rate * math.cos(azimuth_rad)
        vy = mock_rate * math.sin(azimuth_rad)
        
        risk_score = risk_score_calculator.calculate_risk(x_current, y_current, vx, vy)
        
        obj = trajectory_planner.WorldObject(
            obj_id=i, class_id=cls_id, 
            x=x_current, y=y_current, 
            vx=vx, vy=vy, risk_score=risk_score
        )
        world_objects.append(obj)
        
        # --- ANNOTATE FRAME VISUALLY ---
        # Draw bounding box (Red if risk > 10, else Green)
        color = (0, 0, 255) if risk_score > 10 else (0, 255, 0)
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        label = f'{obj.name} | Risk: {risk_score:.1f}'
        cv2.putText(frame, label, (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
    # Show first 3 frames visually in the notebook for confirmation
    if frame_idx < 3:
        ax.clear()
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.axis('off')
        ax.set_title(f'Frame {frame_idx} | YOLO + Sensor Fusion Mock Annotations')
        display(fig)
        clear_output(wait=True)
        
    # 3. Trajectory Planning
    if world_objects:
        time_horizon = 3.0
        predicted_obs = trajectory_planner.predict_obstacle_trajectories(world_objects, time_horizon, dt)
        candidates = trajectory_planner.generate_candidate_trajectories(ego_speed, time_horizon, dt)
        best_traj = None
        min_cost = float('inf')
        for tr in candidates:
            cost = trajectory_planner.evaluate_trajectory_cost(tr, predicted_obs, world_objects)
            if cost < min_cost:
                min_cost = cost
                best_traj = tr
        aeb_triggered = min_cost > 1000000
        y_target = best_traj['offset'] if best_traj else 0.0
        wheelbase = 2.5
        peak_steer = (y_target * 2.0 * math.pi * wheelbase) / (ego_speed * (time_horizon ** 2))
        delta = peak_steer * math.sin(2 * math.pi * (dt / time_horizon))
        steer_angle = round(delta, 4)
        accel = -5.0 if aeb_triggered else 0.0
    else:
        steer_angle = 0.0
        accel = 0.0
        
    timetable_records.append({'Time': round(frame_idx * dt, 3), 'SteeringAngle': steer_angle, 'Acceleration': accel})
    
    frame_idx += 1
    if frame_idx >= 60: # Limit to 60 frames for quicker preview
        break
        
cap.release()
plt.close(fig)
print('✅ Pipeline preview complete!')


Starting pipeline...


Processed 30 frames...


Processed 60 frames...


Processed 90 frames...


Processed 120 frames...


...

Processed 480 frames...


Processed 510 frames...


Processed 540 frames...


Processed 570 frames...


Processed 600 frames...
✅ Pipeline complete!


# 3. Export Controls to CSV and Simulink Script

In [3]:
# Generate pandas DataFrame
df = pd.DataFrame(timetable_records)
csv_path = '/Users/pavanreddy/Epilogue/YOLO_Inference/matlab_data_4/navigation_controls.csv'
df.to_csv(csv_path, index=False)
print(f'Saved Timetable CSV to {csv_path}')

# Export MATLAB/Simulink format (.m script)
m_script = '% Auto-generated by Sensor Fusion & Trajectory Engine\n\n'
m_script += f'time_vector = {df["Time"].tolist()};\n'
m_script += f'steer_vector = {df["SteeringAngle"].tolist()};\n'
m_script += f'accel_vector = {df["Acceleration"].tolist()};\n\n'
m_script += 'SteeringInput = timeseries(steer_vector, time_vector);\n'
m_script += 'AccelerationInput = timeseries(accel_vector, time_vector);\n'
m_script += "ControlTimetable = timetable(seconds(time_vector'), steer_vector', accel_vector', ...\n"
m_script += "    'VariableNames', {'SteeringAngle', 'Acceleration'});\n"

m_path = '/Users/pavanreddy/Epilogue/YOLO_Inference/matlab_data_4/simulink_controls.m'
with open(m_path, 'w') as f:
    f.write(m_script)
print(f'Saved Simulink .m script to {m_path}')

# Show preview
df.head(10)


Saved Timetable CSV to /Users/pavanreddy/Epilogue/YOLO_Inference/matlab_data_4/navigation_controls.csv
Saved Simulink .m script to /Users/pavanreddy/Epilogue/YOLO_Inference/matlab_data_4/simulink_controls.m


,Time,SteeringAngle,Acceleration
0,0.000,0.0122,0.0
1,0.033,0.0000,0.0
2,0.067,0.0000,0.0
3,0.100,0.0000,0.0
4,0.133,0.0000,0.0
5,0.167,0.0000,0.0
6,0.200,0.0000,0.0
7,0.233,0.0304,-5.0
8,0.267,0.0304,-5.0
9,0.300,0.0304,-5.0
